In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

project_path = "/content/drive/MyDrive/Fraud-Detection-Project"
os.chdir(project_path)

print(os.getcwd())

/content/drive/MyDrive/Fraud-Detection-Project


In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
df = pd.read_csv("data/Base.csv")

X = df.drop(columns=["fraud_bool"])
y = df["fraud_bool"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [ ]:
xgb_model = XGBClassifier(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    random_state=42
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=skf,
    scoring="roc_auc",
    n_jobs=-1
)

pr_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=skf,
    scoring="average_precision",
    n_jobs=-1
)

print("Cross-Validated ROC-AUC:", roc_scores.mean())
print("Cross-Validated PR-AUC:", pr_scores.mean())

Cross-Validated ROC-AUC: 0.8809010033725706
Cross-Validated PR-AUC: 0.14732417120442026


In [ ]:
param_dist = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0]
}

In [ ]:
xgb_model = XGBClassifier(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    random_state=42
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

random_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_dist,
    n_iter=10,              # try 10 random combinations
    scoring="average_precision",  # optimize PR-AUC (important!)
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [11]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               Index(['income', 'name_email_similarity', 'prev_address_months_count',
       'current_address_months_count', 'customer_age', 'days_since_request',
       'intended_balcon_am...
                                                            multi_strategy=None,
                                                            n_estimators=None,
                                                            n_jobs=None,
                                                            num_parallel_tree=None, ...))]),
                   n_jobs=-1,
                   param_distributions={'classifier__colsample_bytree': [0.8,
                                                                         1.0],
                                        'classifier__learning_rate': [0.01,
                                                                      0.05,
                                                                      0.1],
                                        'classifier__max_depth': [3, 5, 7],
                                        'classifier__n_estimators': [100, 200,
                                                                     300],
                                        'classifier__subsample': [0.8, 1.0]},
                   random_state=42, scoring='average_precision', verbose=2)

In [12]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest PR-AUC (CV):")
print(random_search.best_score_)

Best Parameters:
{'classifier__subsample': 1.0, 'classifier__n_estimators': 300, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.1, 'classifier__colsample_bytree': 1.0}

Best PR-AUC (CV):
0.17420542861155686


In [13]:
best_model = random_search.best_estimator_

y_proba = best_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score, average_precision_score

print("Tuned ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Tuned PR-AUC:", average_precision_score(y_test, y_proba))

Tuned ROC-AUC: 0.8986631211803822
Tuned PR-AUC: 0.17994167384646995


In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

best_threshold = 0
best_f1 = 0

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold (F1):", best_threshold)
print("Best F1:", best_f1)

Best Threshold (F1): 0.9208093
Best F1: 0.24353628023352794


In [ ]:
from sklearn.metrics import confusion_matrix

def fraud_cost(y_true, y_pred, fn_cost=10, fp_cost=1):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn * fn_cost + fp * fp_cost

best_cost = float("inf")
best_cost_threshold = 0

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    cost = fraud_cost(y_test, y_pred_t)

    if cost < best_cost:
        best_cost = cost
        best_cost_threshold = t

print("Best Threshold (Cost):", best_cost_threshold)
print("Minimum Cost:", best_cost)

Best Threshold (Cost): 0.88854784
Minimum Cost: 17694


In [ ]:
import joblib
joblib.dump(best_model, "models/tuned_xgboost.pkl")

['models/tuned_xgboost.pkl']